# House Price Prediction - Inference

Load trained model and make predictions on new data from Supabase.


In [ ]:
# Imports
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import pickle
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent.parent))
from pipeline.supabase_handler import fetch_csv_from_supabase

print("✅ Imports successful")

## Step 1: Load Trained Model


In [ ]:
# Find and load latest model
model_dir = Path('saved_models')

if not model_dir.exists():
    print("❌ No saved_models directory found!")
    print("Run training first: python models/training/train.py")
else:
    # Find latest model
    model_files = list(model_dir.glob('*.joblib'))
    if not model_files:
        print("❌ No trained models found!")
    else:
        latest_model = sorted(model_files)[-1]
        model_name = latest_model.stem
        
        print(f"Loading model: {model_name}")
        model = joblib.load(latest_model)
        
        # Load metadata
        meta_path = model_dir / f"{model_name}_meta.pkl"
        with open(meta_path, 'rb') as f:
            metadata = pickle.load(f)
        
        print(f"✓ Model loaded successfully")
        print(f"  Type: {metadata['model_type']}")
        print(f"  Features: {len(metadata['features'])}")
        print(f"  Test R²: {metadata['metrics']['test_r2']:.4f}")
        print(f"  Test RMSE: {metadata['metrics']['rmse']:.4f} billion VND")

## Step 2: Load Data from Supabase


In [ ]:
# Load data
print("Loading data from Supabase...")
df = fetch_csv_from_supabase("alonhadat_features")
print(f"✓ Loaded {len(df)} records")

# Check shape
print(f"\nData shape: {df.shape}")

## Step 3: Prepare Features


In [ ]:
# Get features from metadata
features = metadata['features']

# Check available features
available_features = [col for col in features if col in df.columns]
missing_features = [col for col in features if col not in df.columns]

print(f"Using {len(available_features)}/{len(features)} features")
if missing_features:
    print(f"⚠️  Missing: {missing_features}")

# Prepare feature matrix
X = df[available_features].copy()
X = X.fillna(X.median())

print(f"\n✓ Features prepared")
print(f"  Shape: {X.shape}")

## Step 4: Make Predictions


In [ ]:
# Make predictions
print("Making predictions...")
y_pred = model.predict(X)
print(f"✓ Made {len(y_pred)} predictions")

# Add to dataframe
df['predicted_price_billion_vnd'] = y_pred

# Display samples
print("\nSample predictions:")
df[['area_m2', 'distance_to_center_km', 'predicted_price_billion_vnd']].head(10)

## Step 5: Compare with Actual Prices (if available)


In [ ]:
# Check if actual prices available
price_col = None
if 'price_billion_vnd' in df.columns:
    price_col = 'price_billion_vnd'
elif 'price' in df.columns:
    price_col = 'price'

if price_col:
    df['actual_price_billion_vnd'] = df[price_col]
    df['error_billion_vnd'] = df['predicted_price_billion_vnd'] - df['actual_price_billion_vnd']
    df['error_pct'] = (df['error_billion_vnd'] / df['actual_price_billion_vnd']) * 100
    
    # Calculate statistics
    mae = np.abs(df['error_billion_vnd']).mean()
    rmse = np.sqrt((df['error_billion_vnd'] ** 2).mean())
    
    print("Prediction Error Statistics:")
    print(f"  Mean Absolute Error: {mae:.4f} billion VND")
    print(f"  Root Mean Squared Error: {rmse:.4f} billion VND")
    print(f"  Median Error: {np.median(df['error_billion_vnd']):.4f} billion VND")
    print(f"  Std Dev: {df['error_billion_vnd'].std():.4f} billion VND")
    print(f"  Mean Error %: {df['error_pct'].mean():.2f}%")
    
    # Show examples
    print("\nSample predictions vs actual:")
    sample_cols = ['area_m2', 'actual_price_billion_vnd', 'predicted_price_billion_vnd', 'error_pct']
    print(df[sample_cols].head(10).to_string(index=False))
else:
    print("No actual prices available in data")

## Step 6: Visualize Results


In [ ]:
if price_col:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Actual vs Predicted scatter
    axes[0, 0].scatter(df['actual_price_billion_vnd'], df['predicted_price_billion_vnd'], alpha=0.5)
    min_price = min(df['actual_price_billion_vnd'].min(), df['predicted_price_billion_vnd'].min())
    max_price = max(df['actual_price_billion_vnd'].max(), df['predicted_price_billion_vnd'].max())
    axes[0, 0].plot([min_price, max_price], [min_price, max_price], 'r--', lw=2)
    axes[0, 0].set_xlabel('Actual Price (billion VND)')
    axes[0, 0].set_ylabel('Predicted Price (billion VND)')
    axes[0, 0].set_title('Actual vs Predicted Prices')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Error distribution
    axes[0, 1].hist(df['error_billion_vnd'], bins=30, edgecolor='black', alpha=0.7)
    axes[0, 1].axvline(df['error_billion_vnd'].mean(), color='r', linestyle='--', 
                        label=f"Mean: {df['error_billion_vnd'].mean():.3f}")
    axes[0, 1].set_xlabel('Prediction Error (billion VND)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Distribution of Errors')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Error % distribution
    axes[1, 0].hist(df['error_pct'], bins=30, edgecolor='black', alpha=0.7)
    axes[1, 0].axvline(df['error_pct'].mean(), color='r', linestyle='--',
                        label=f"Mean: {df['error_pct'].mean():.1f}%")
    axes[1, 0].set_xlabel('Prediction Error (%)')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Distribution of Error %')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Prediction vs Area
    axes[1, 1].scatter(df['area_m2'], df['error_pct'], alpha=0.5)
    axes[1, 1].axhline(0, color='r', linestyle='--')
    axes[1, 1].set_xlabel('Area (m²)')
    axes[1, 1].set_ylabel('Error %')
    axes[1, 1].set_title('Error by Property Area')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    # Just show predictions
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(df['predicted_price_billion_vnd'], bins=30, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Predicted Price (billion VND)')
    ax.set_ylabel('Frequency')
    ax.set_title('Distribution of Predicted Prices')
    ax.grid(True, alpha=0.3)
    plt.show()

## Step 7: Save Predictions


In [ ]:
# Create predictions directory
pred_dir = Path('data/predictions')
pred_dir.mkdir(parents=True, exist_ok=True)

# Save predictions
output_file = pred_dir / 'predictions_latest.csv'
df.to_csv(output_file, index=False)

print(f"✓ Saved predictions: {output_file}")
print(f"  Records: {len(df)}")
print(f"  Size: {output_file.stat().st_size / 1024 / 1024:.2f} MB")

print(f"\n✅ Inference complete!")
print(f"   Predictions saved to: {output_file}")

## Summary Statistics


In [ ]:
# Summary
print("\n" + "="*60)
print("PREDICTION SUMMARY")
print("="*60)
print(f"Model: {metadata['model_type']}")
print(f"Total Records Predicted: {len(df)}")
print(f"\nPredicted Price Range:")
print(f"  Min: {df['predicted_price_billion_vnd'].min():.4f} billion VND")
print(f"  Max: {df['predicted_price_billion_vnd'].max():.4f} billion VND")
print(f"  Mean: {df['predicted_price_billion_vnd'].mean():.4f} billion VND")
print(f"  Median: {df['predicted_price_billion_vnd'].median():.4f} billion VND")

if price_col:
    print(f"\nModel Performance:")
    print(f"  MAE: {np.abs(df['error_billion_vnd']).mean():.4f} billion VND")
    print(f"  RMSE: {np.sqrt((df['error_billion_vnd'] ** 2).mean()):.4f} billion VND")
    print(f"  Mean Error %: {df['error_pct'].mean():.2f}%")

print("="*60)